# Step 9 · Validate the native V4 release inputs

This notebook checks taxonomy and article links, screens, and the run-level pair evidence before Explorer packaging.


In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, log_step
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('09_validate_release', RUN_ID)
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-02 21:44 INFO Notebook session started at 2026-11-02T21:44:55.834804+00:00


2026-11-02 21:44 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 21:44 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/09_validate_release.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/09_validate_release.log


## Step 1 · Load the current run evidence in full mode, or the staged snapshot otherwise

In [2]:
import json
import yaml
import pandas as pd
from IPython.display import display
from pathogen_atlas_v4.pubmed_client import literature_cache_key
from pathogen_atlas_v4.release import validate_pair_release
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text(encoding='utf-8'))
RUN_ID = str(CONFIG['run']['run_id'])
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
NATIVE_EVIDENCE_PATH = RUN_DIR / 'evidence/pathogen_host_evidence.parquet'
if str(CONFIG['run'].get('selection_mode', '')).casefold() != 'full':
    raise RuntimeError('Native V4 release validation requires a completed full run.')
if not NATIVE_EVIDENCE_PATH.exists():
    raise FileNotFoundError(f'Run notebook 06 first; pair evidence is missing: {NATIVE_EVIDENCE_PATH}')
STAGING_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/'))[-1]
STAGING_TAXONOMY_ROOT = STAGING_ROOT / 'taxonomy'
taxonomy_snapshot = STAGING_TAXONOMY_ROOT.parent.name
cache_key = literature_cache_key(query_version=CONFIG['acquisition']['query_strategy_version'],
    pathogen_literature_query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
    taxonomy_snapshot=taxonomy_snapshot)
CACHE_DIR = ROOT / 'data/v4/cache/literature' / cache_key
pathogens = pd.read_parquet(RUN_DIR / 'taxonomy/pathogens.parquet')
hosts = pd.read_parquet(STAGING_TAXONOMY_ROOT / 'hosts.parquet')
articles = pd.read_parquet(CACHE_DIR / 'tables/articles.parquet')
evidence = pd.read_parquet(NATIVE_EVIDENCE_PATH)
matches = pd.read_parquet(CACHE_DIR / 'tables/pathogen_screen_matches.parquet')
resolved_record = json.loads((RUN_DIR / 'resolved_config.json').read_text(encoding='utf-8'))
selected_pair_keys = {(str(pair['pathogen_id']), str(pair['host_id'])) for pair in resolved_record['pairs']}
if not matches.empty:
    match_index = pd.MultiIndex.from_frame(matches[['pathogen_id', 'host_id']].astype(str))
    matches = matches.loc[match_index.isin(selected_pair_keys)].copy()
print('Validation source: native V4 run', RUN_ID)
print('Pathogens:', f'{len(pathogens):,}', '| hosts:', len(hosts), '| articles:', f'{len(articles):,}', '| pair evidence:', f'{len(evidence):,}')
LOG.info('Release validation inputs loaded: run=%s evidence=%d', RUN_ID, len(evidence))


2026-11-02 21:44 INFO Release validation inputs loaded: run=pilot-v1 evidence=14


Validation source: native V4 run pilot-v1
Pathogens: 142,201 | hosts: 9 | articles: 14 | pair evidence: 14


## Step 2 · Run foreign-key and evidence-provenance validation

In [3]:
validate_pair_release(pathogens, hosts, articles, evidence)
print('Pair-release integrity checks passed.')
LOG.info('Pair release foreign keys and evidence-provenance checks passed')

2026-11-02 21:44 INFO Pair release foreign keys and evidence-provenance checks passed


Pair-release integrity checks passed.


## Step 3 · Summarize provenance and pair outcomes

In [4]:
display(evidence.groupby(['provenance_status','decision'], dropna=False).size().rename('records').to_frame())
print('Unique supported pathogen–host pairs:', evidence.loc[evidence.decision.eq('supported'), ['pathogen_id','host_id']].drop_duplicates().shape[0])
LOG.info('Evidence outcomes: %s', evidence.groupby(['provenance_status','decision']).size().to_dict())

records
provenance_status decision            
system1_jev       supported          5
                  unsupported        9

2026-11-02 21:44 INFO Evidence outcomes: {('system1_jev', 'supported'): 5, ('system1_jev', 'unsupported'): 9}


Unique supported pathogen–host pairs: 3


## Step 4 · Build pair-level literature metrics

In [5]:
pair_summary = matches.groupby(['pathogen_id', 'host_id'], dropna=False).agg(
    n_pair_search_records=('pair_search_count', 'first'),
    n_sampled_pmids=('pmid', 'nunique'),
    n_abstracts_available=('abstract_available', 'sum'),
).reset_index()
supported = evidence.loc[evidence.decision.astype(str).eq('supported')]
supported_counts = supported.groupby(['pathogen_id', 'host_id']).pmid.nunique().rename('n_supported_pmids').reset_index()
pair_summary = pair_summary.merge(supported_counts, on=['pathogen_id', 'host_id'], how='left')
pair_summary['n_supported_pmids'] = pair_summary.n_supported_pmids.fillna(0).astype('int64')
display(pair_summary.sort_values('n_supported_pmids', ascending=False).head(25))
print('Pair-aware sample pairs:', f'{len(pair_summary):,}', '| supported PMIDs:', int(pair_summary.n_supported_pmids.sum()))
LOG.info('Pair evidence metrics reviewed: evidence=%d', len(evidence))


,pathogen_id,host_id,n_pair_search_records,n_sampled_pmids,n_abstracts_available,n_supported_pmids
9,viruses:fa9873798bea5617f432,ncbi:4530,4,4,4,3
5,viruses:976f27c4e726cd4225b8,ncbi:4530,1,1,1,1
8,viruses:f77114fb0e9d224e3584,ncbi:4530,1,1,1,1
0,viruses:14a1f128bfc11dc44854,ncbi:4530,1,1,1,0
1,viruses:45559d9fda37873e6224,ncbi:4530,1,1,1,0
2,viruses:5e24f1dfe516de42c1a4,ncbi:4530,1,1,1,0
3,viruses:71279d492d6f0d6c2d14,ncbi:4530,2,2,2,0
4,viruses:89338c2d02f6e298bca3,ncbi:4530,1,1,1,0
6,viruses:9b1f599169208cb01533,ncbi:4530,1,1,1,0
7,viruses:dfeae45bf61237db36f0,ncbi:4530,1,1,1,0


2026-11-02 21:44 INFO Pair evidence metrics reviewed: evidence=14


Pair-aware sample pairs: 10 | supported PMIDs: 5


## Step 5 · Release decision

In [6]:
BUILD_NATIVE_RELEASE = False
if BUILD_NATIVE_RELEASE:
    raise RuntimeError('Build a native release only after pair-aware acquisition, disease-gate review, and profile completeness checks.')
print('Validation only. No tables were promoted or labelled as a native V4 release.')
LOG.info('Native release build skipped')

2026-11-02 21:44 INFO Native release build skipped


Validation only. No tables were promoted or labelled as a native V4 release.
